In [143]:
import pulp
import pandas as pd

In [144]:
products_file = "../data/availableProducts.csv"

products_df = pd.read_csv(products_file)
list(products_df["Product name"].unique())



['Pingui',
 'Maxi King',
 'Choco Fresh',
 'Paradiso',
 'Milchschnitte',
 'Riegel',
 'Schokobons',
 'Bueno',
 'Happy Hippo',
 'Cards',
 'Überraschung']

In [145]:
number_of_doors = 24
number_of_products = 10

Doors = list(range(1, number_of_doors+1))
Products = list(products_df["Product name"].unique())


max_product_items = 24*2

problem = pulp.LpProblem("AdventCalendarOptimization", pulp.LpMaximize)

In [146]:
number_of_prodcuts_in_door = pulp.LpVariable.dicts("Items_per_door", (Doors, Products), lowBound=0, upBound=1, cat=pulp.LpInteger)

product_used = pulp.LpVariable.dicts("Product_Used", Products, cat=pulp.LpBinary)

In [147]:
problem += pulp.lpSum(product_used[product] for product in Products), "Total product coverage"

In [148]:
for product in Products:
    total_items = pulp.lpSum(number_of_prodcuts_in_door[door][product] for door in Doors)

    problem += total_items <= max_product_items * product_used[product], f"Linking_constraint_{product}"
    problem += total_items >= 1 * product_used[product], f"Linking_constraint_ensure_1_{product}"


In [149]:
MAX_ITEMS_PER_DOOR = 3
MIN_ITEMS_PER_DOOR = 2

for door in Doors:
    problem += pulp.lpSum(number_of_prodcuts_in_door[door][product] for product in Products) <= MAX_ITEMS_PER_DOOR

for door in Doors:
    problem += pulp.lpSum(number_of_prodcuts_in_door[door][product] for product in Products) >= MIN_ITEMS_PER_DOOR

In [150]:
MAX_PRODUCTS = 20

minimum_items_per_product = pd.Series(products_df["minimum package quantity"].values, index=products_df["Product name"]).to_dict()

minimum_items_per_product["Überraschung"] = 2

for product in Products:
    problem += pulp.lpSum(number_of_prodcuts_in_door[door][product] for door in Doors) <= MAX_PRODUCTS
    problem += pulp.lpSum(number_of_prodcuts_in_door[door][product] for door in Doors) >= minimum_items_per_product[product]


In [151]:
problem.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/leander-merbecks/Documents/git-repositories/projects/ferrerokinderAdventCalendar/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/bb08542320364b1aac59bc7c7e3260d4-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/bb08542320364b1aac59bc7c7e3260d4-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 97 COLUMNS
At line 2265 RHS
At line 2358 BOUNDS
At line 2634 ENDATA
Problem MODEL has 92 rows, 275 columns and 1606 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 11 - 0.00 seconds
Cgl0003I 11 fixed, 0 tightened bounds, 0 strengthened rows, 0 substitutions
Cgl0004I processed model has 35 rows, 264 columns (264 integer (264 of which binary)) and 528 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of

1

In [152]:
print(f"Optimizer status {pulp.LpStatus[problem.status]}")
print(f"Product coverage {pulp.value(problem.objective)/len(Products)}")

Optimizer status Optimal
Product coverage 1.0


In [153]:
for door in Doors:
    items_in_this_door = []
    total_items = 0

    for product in Products:
        items = pulp.value(number_of_prodcuts_in_door[door][product])

        if items > 0:
            items_in_this_door.append(f"{product}: {int(items)}")
        
        total_items += items
    
    if total_items > 0:
        print(f"Door {door} (total {int(total_items)}): {', '.join(items_in_this_door)}")

Door 1 (total 3): Paradiso: 1, Schokobons: 1, Cards: 1
Door 2 (total 2): Riegel: 1, Schokobons: 1
Door 3 (total 2): Pingui: 1, Schokobons: 1
Door 4 (total 3): Riegel: 1, Schokobons: 1, Bueno: 1
Door 5 (total 3): Choco Fresh: 1, Paradiso: 1, Happy Hippo: 1
Door 6 (total 3): Maxi King: 1, Choco Fresh: 1, Schokobons: 1
Door 7 (total 3): Riegel: 1, Schokobons: 1, Cards: 1
Door 8 (total 3): Choco Fresh: 1, Schokobons: 1, Happy Hippo: 1
Door 9 (total 3): Riegel: 1, Schokobons: 1, Bueno: 1
Door 10 (total 2): Paradiso: 1, Schokobons: 1
Door 11 (total 3): Choco Fresh: 1, Schokobons: 1, Happy Hippo: 1
Door 12 (total 3): Riegel: 1, Schokobons: 1, Happy Hippo: 1
Door 13 (total 3): Milchschnitte: 1, Riegel: 1, Cards: 1
Door 14 (total 3): Pingui: 1, Schokobons: 1, Bueno: 1
Door 15 (total 3): Riegel: 1, Cards: 1, Überraschung: 1
Door 16 (total 3): Maxi King: 1, Riegel: 1, Schokobons: 1
Door 17 (total 3): Schokobons: 1, Bueno: 1, Überraschung: 1
Door 18 (total 3): Milchschnitte: 1, Riegel: 1, Bueno: 1

In [154]:
for product in Products:
    total_occurences_expression = pulp.lpSum(number_of_prodcuts_in_door[door][product] for door in Doors)
    print(f"{product} occurs {int(pulp.value(total_occurences_expression))} times")

Pingui occurs 4 times
Maxi King occurs 3 times
Choco Fresh occurs 5 times
Paradiso occurs 4 times
Milchschnitte occurs 4 times
Riegel occurs 10 times
Schokobons occurs 20 times
Bueno occurs 6 times
Happy Hippo occurs 5 times
Cards occurs 5 times
Überraschung occurs 2 times
